In [1]:
import json
from pathlib import Path
from collections import defaultdict

In [2]:
BASE_DIR = Path.cwd().parent

CATEGORY = "test"
COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

# The master file containing all subfigures and question structures
ORIGINAL_STATE_FILE = BASE_DIR / f"submission_finetuning_{CATEGORY}_state.json"
FINAL_SUBMISSION_PATH = BASE_DIR / f"submission_finetuning_{CATEGORY}.json"

# The 4 specific state files containing your generated answers
STATE_FILES = [
    BASE_DIR / f"submission_finetuning_{CATEGORY}__factoid_state.json",
    BASE_DIR / f"submission_finetuning_{CATEGORY}__paragraph_state.json",
    BASE_DIR / f"submission_finetuning_{CATEGORY}__list_state.json",
    BASE_DIR / f"submission_finetuning_{CATEGORY}__yes_no_state.json",
]

In [3]:
def build_validated_submission():
    # 1. Load all generated answers into a single lookup dictionary
    # Structure: answers_lookup[(sample_id, sub_fig, question_text)] = answer_text
    answers_lookup = {}
    seen_keys = set()

    print("Loading partial states...")
    for state_file in STATE_FILES:
        if not state_file.exists():
            print(f"  [!] Warning: {state_file.name} not found. Skipping.")
            continue

        with open(state_file, "r") as f:
            partial_state = json.load(f)

        for sample_id, sub_figs in partial_state.items():
            for sub_fig, questions in sub_figs.items():
                # Handle both dict and list formats securely
                if isinstance(questions, dict):
                    items_to_process = questions.items()
                elif isinstance(questions, list):
                    items_to_process = [(q["question"], q["answer"]) for q in questions]
                else:
                    continue

                for q_text, ans_text in items_to_process:
                    unique_key = (sample_id, sub_fig, q_text)

                    if unique_key not in seen_keys:
                        seen_keys.add(unique_key)
                        answers_lookup[unique_key] = ans_text

    print(f"✅ Loaded {len(answers_lookup)} unique answers from state files.\n")

    # 2. Parse the ground truth directory to build the exact expected submission structure
    print("Scanning dataset directory to build guaranteed submission structure...")
    json_files = list(CASE_DIR.rglob("*.json"))

    submission = []
    missing_answers_count = 0
    expected_questions_count = 0

    # Dictionary to track missing counts by answer_type
    missing_per_type = defaultdict(int)

    for json_file in json_files:
        fullpath = str(json_file)
        if (
            "content.json" in json_file.name
            or "images" not in fullpath
            or ".vscode" in fullpath
        ):
            continue

        with open(json_file, "r") as f:
            data = json.load(f)

        sample_id = data["sample_id"]
        vqa_data = data.get("vqa", {})

        sample_output_vqa = {}

        for sub_fig, q_list in vqa_data.items():
            sample_output_vqa[sub_fig] = []

            for q_obj in q_list:
                expected_questions_count += 1
                q_text = q_obj.get("question") or q_obj.get("questions")
                ans_type = q_obj.get("answer_type", "Unknown")

                # Retrieve the answer from our lookup, fallback to empty string if missing
                unique_key = (sample_id, sub_fig, q_text)
                ans_text = answers_lookup.get(unique_key, "")

                if not ans_text:
                    missing_answers_count += 1
                    missing_per_type[ans_type] += 1
                    print(
                        f"  [!] Missing [{ans_type}] Answer for {sample_id} | {sub_fig} | Q: {q_text[:30]}..."
                    )

                # Append strictly exactly what the evaluator expects
                sample_output_vqa[sub_fig].append(
                    {
                        "question": q_text,
                        "question_type": q_obj.get("question_type", ""),
                        "answer_type": ans_type,
                        "answer": ans_text,
                    }
                )

        submission.append({"sample_id": sample_id, "vqa": sample_output_vqa})

    # 3. Save the final submission
    with open(FINAL_SUBMISSION_PATH, "w") as f:
        json.dump(submission, f, indent=2)

    # 4. Print detailed summary
    print(f"\n✅ Validation & Merge Complete!")
    print(f"📊 Expected Questions: {expected_questions_count}")
    print(f"📊 Answers Found: {expected_questions_count - missing_answers_count}")
    print(f"📊 Answers Missing Total: {missing_answers_count}")

    if missing_answers_count > 0:
        print("   Breakdown of missing answers by type:")
        for ans_type, count in missing_per_type.items():
            print(f"    - {ans_type}: {count}")

    print(
        f"\n💾 Saved strictly validated submission ({len(submission)} samples) to {FINAL_SUBMISSION_PATH.name}"
    )

In [4]:
build_validated_submission()

Loading partial states...
✅ Loaded 1783 unique answers from state files.

Scanning dataset directory to build guaranteed submission structure...
  [!] Missing [Factoid] Answer for atomic-layer-etching/experimental-usecase/12/fig_3 | b | Q: How does the film thickness ch...
  [!] Missing [Paragraph] Answer for atomic-layer-etching/experimental-usecase/12/fig_3 | b | Q: Why is there a drop in thickne...
  [!] Missing [Yes/No] Answer for atomic-layer-etching/experimental-usecase/12/fig_3 | b | Q: Does the linear regime indicat...
  [!] Missing [Paragraph] Answer for atomic-layer-etching/experimental-usecase/12/fig_3 | b | Q: How does the etch profile obse...
  [!] Missing [Factoid] Answer for atomic-layer-etching/experimental-usecase/12/fig_3 | b | Q: What is the Etch Per Cycle (EP...
  [!] Missing [Yes/No] Answer for atomic-layer-etching/experimental-usecase/12/fig_3 | b | Q: Does the inset show the step b...
  [!] Missing [Factoid] Answer for atomic-layer-etching/experimental-usecase/12